<a href="https://colab.research.google.com/github/emmanuelle-bul/AI_Exercises/blob/main/TextFileToChromaDbFunc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q chromadb openai

1. **Каква векторна база да използваме?** -> Chromadb

2. **Какъв ембединг модел трябва да използваме?** -> OpenAI Embedding model - много важно!!! Ако сме напълнили вече базта с милиони вектори и в даден момент решим да сменим модела => изключително скъпа и времеемка инициатива => за всеки запис от базата се чете текста и на ново се генерира embedding вектор

**!!! Embedding вектор, генриран с AI модел А, не може директно да бъде сравнен с embedding вектор, генериран с AI модел В.
Не можем да сравняваме embedding вектори, генерирани от различни модели, дори не можем да сравняваме вектори, генерирани със small и large моделите. Те са несъвместими.**


In [ ]:
from chromadb import PersistentClient
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction
from google.colab import userdata
from pathlib import Path
from openai import OpenAI
from pydantic import BaseModel

openai_api_key = userdata.get("OPENAI_API_KEY")
openai_client = OpenAI(api_key=openai_api_key)

chromadb_client = PersistentClient("/content/chromadb")
# създаваме колекция за файла - т.е. нова db
chromadb_collection = chromadb_client.get_or_create_collection(
    name="books",
    embedding_function=OpenAIEmbeddingFunction(api_key=openai_api_key, model_name="text-embedding-3-large")
)

# дефинираме функция с входни параметри и дефолтни стойности, която не връща резултат (-> None)
def index_file(path_to_file: Path, chunk_size: int=5, chunk_overlap_size: int=2, batch_size: int=100) -> None:
  with path_to_file.open() as file_content:
    file_lines = file_content.readlines()    # връща списък от редове
    file_lines = [fl.strip() for fl in file_lines] # премахва \t и \n в началото и в края на реда
    file_lines = [fl for fl in file_lines if fl != ''] # премахва празните редове

    chunks = []
    line_idx = 0
    while line_idx < len(file_lines):
      start = line_idx
      end = line_idx + chunk_size

      current = '\n'.join(file_lines[start:end])
      chunks.append({"text": current, "book": path_to_file.stem, "rel_line_start": start+1, "rel_line_end": end})
      line_idx = end - chunk_overlap_size

    chunk_idx = 0
    while chunk_idx < len(chunks):
      start = chunk_idx
      end = chunk_idx + batch_size
      current_batch = chunks[start:end]

      chromadb_collection.add(
          ids=[f"{x['book']}_{x['rel_line_start']}_{x['rel_line_end']}" for x in current_batch],
          documents=[x['text'] for x in current_batch],
          metadatas=[{"book": i['book'], "rel_line_start": i['rel_line_start'], "rel_line_end": i['rel_line_end']} for i in current_batch]
      )

      chunk_idx = end



In [ ]:
index_file(Path("/content/Noev_kovcheg.txt") )

In [ ]:
results = chromadb_collection.query(
    query_texts="мъртъв"      #смърт, умр, удавник
)

In [ ]:
results

{'ids': [['Noev_kovcheg_46_50',
   'Noev_kovcheg_235_239',
   'Noev_kovcheg_43_47',
   'Noev_kovcheg_433_437',
   'Noev_kovcheg_1_5',
   'Noev_kovcheg_16_20',
   'Noev_kovcheg_430_434',
   'Noev_kovcheg_265_269',
   'Noev_kovcheg_1069_1073',
   'Noev_kovcheg_22_26']],
 'embeddings': None,
 'documents': [['Мао Дзедун бе много смален подир смъртта си, не по-висок от пет– или шестгодишно дете. В едната си ръка държеше отпечатък от своя собствен портрет, а с другата си ръка го приглаждаше постоянно, мъчейки се напразно да изглади гънките му. Портретът бе мачкан и тъпкан с крака. Собственикът му по всяка вероятност го бе намерил на улицата.\nМъртвият човек се натъжаваше още повече, като виждаше, че не може да изглади гънките на смачкания си отпечатък, нито пък да изтрие петната по него. Освен мътни петна по лицето имаше следи от обуща и боси ходила. Тези, които бяха мачкали отпечатъка, изглежда, че го бяха и подритвали и тъпкали.\nДвамата се гледахме, без да си говорим, както не си говорехм

In [ ]:
class QuoteLookupArgs(BaseModel):
  query: str
  # top_n: int

In [ ]:
QuoteLookupArgs.model_json_schema()

{'properties': {'query': {'title': 'Query', 'type': 'string'}},
 'required': ['query'],
 'title': 'QuoteLookupArgs',
 'type': 'object'}

In [ ]:
# описанието да бъде с императивен изказ - работи по-добре, отколкото "този тул може да бъде извика за ...."
TOOLS = [
    {
        "type": "function",
        "name": "quote_lookup",
        "description": "Call this tool to lookup a quote from the internal collection of books that corresponds semantically to a given natural language query",
        "parameters": {
            **QuoteLookupArgs.model_json_schema(),
            "additionalProperties": False
        },
        "strict": True
    }
]

In [ ]:
TOOLS

[{'type': 'function',
  'name': 'quote_lookup',
  'description': 'Call this tool to lookup a quote from the internal collection of books that corresponds semantically to a given natural language query',
  'parameters': {'properties': {'query': {'title': 'Query', 'type': 'string'}},
   'required': ['query'],
   'title': 'QuoteLookupArgs',
   'type': 'object',
   'additionalProperties': False},
  'strict': True}]

In [ ]:
chromadb_collection.query(query_texts="мъртъв")

# в резултата 'ids' е матрица - за всеки подаден текст се връща списък с отговори => ако имаме само 1 текст за query_texts => ids[0][]  (от матрицата вземаме единствения списък)

{'ids': [['Noev_kovcheg_46_50',
   'Noev_kovcheg_235_239',
   'Noev_kovcheg_43_47',
   'Noev_kovcheg_433_437',
   'Noev_kovcheg_1_5',
   'Noev_kovcheg_16_20',
   'Noev_kovcheg_430_434',
   'Noev_kovcheg_265_269',
   'Noev_kovcheg_1069_1073',
   'Noev_kovcheg_22_26']],
 'embeddings': None,
 'documents': [['Мао Дзедун бе много смален подир смъртта си, не по-висок от пет– или шестгодишно дете. В едната си ръка държеше отпечатък от своя собствен портрет, а с другата си ръка го приглаждаше постоянно, мъчейки се напразно да изглади гънките му. Портретът бе мачкан и тъпкан с крака. Собственикът му по всяка вероятност го бе намерил на улицата.\nМъртвият човек се натъжаваше още повече, като виждаше, че не може да изглади гънките на смачкания си отпечатък, нито пък да изтрие петната по него. Освен мътни петна по лицето имаше следи от обуща и боси ходила. Тези, които бяха мачкали отпечатъка, изглежда, че го бяха и подритвали и тъпкали.\nДвамата се гледахме, без да си говорим, както не си говорехм

In [ ]:
import json

def quote_lookup_tool_handler(raw_args: str):
  typed_args = QuoteLookupArgs.model_validate(json.loads(raw_args))  # взимаме json и го парсваме към обекта

  search_results = chromadb_collection.query(query_texts=[typed_args])

  count = len(search_results["ids"][0]) # вземаме списъка с ids

  output = []

  for i in range(count):
    output.append(f"Match #{i + 1}:")

    book = search_results["metadatas"][0][i]["book"]
    output.append(f"Book: {book}")

    quote = search_results["documents"][0][i]
    output.append(f"Quote: {quote}")

    output.append("")
  return "\n".join(output)

# срещу името на инструмента записваме функция, която ще бъде изпълнена, когато инструмента е извикан
TOOL_HANDLERS = {
    "quote_lookup": quote_lookup_tool_handler
}

In [ ]:
quote_lookup_tool_handler('{"query":"вдъхновяващи или ключови цитати от книгата, която е в директорията и колекцията books"}')

ValueError: Expected document to be a str, got query='вдъхновяващи или ключови цитати от книгата, която е в директорията и колекцията books' in query.

In [ ]:
temp_response = openai_client.responses.create(
    model="gpt-5-mini",
    input=[
        {"role": "system", "content": "You are a helpful assitant for book analysis and quote finding. The user will enter a given topic and you need to use the \"quote_lookup\" tool to fing relevant quotes from the internal collection of books. Use it multiple times with different (similar) variations of the original user query. "},
        {"role": "user", "content": f"Намери ми релевантни цитати, които са вързани с книгата в текущата директория и е заредене в {chromadb_collection}"}
    ],
    tools=TOOLS
)

In [ ]:
temp_response

Response(id='resp_01f997735c535ce3006ac796cd6e3087d0a11d6df14731ae24', created_at=1791465165.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5-mini-2025-08-07', object='response', output=[ResponseReasoningItem(id='rs_01f997735c535ce3006ac796cdc79887d080e0cac18750191a', summary=[], type='reasoning', content=[], encrypted_content='gAAAAABqx5bR8vj1Sh5Oj4Xbi_TxTBa4P4oNiJrYEoLYg35EACc_g39x2HQe9J9rYz9Dq8akPu-b4d0dS2_jN3Uk9Z0JEUj-cEQb8Fj25ek7qITYpjk2AWtJGUcovQeMY9IRK9ecz5klI0sOmZcwUddLdkpPGLHvIC8iOSPg9zq144z5M9uTIfIuWyGpJuOnDYJH9zswRDAGihBS9KHdgDUkAcbhTGV6XxEChaqmRxiMs96RU8qy8uLu2Xcx9EYTR16RGCwDGjtkeNbT99zOdq4WHr7_1ZvWmYjJUa25TVZjLTzq3IU3TOl7NNNXWB-yFQd4ZhQvGIlER18ksS0TuNclg5ENTXZrgbDuFBk5NZfZFYHtk2DGoeRxnr9mQ7HvBdi4LStjDt-lK8raJe585uz-bjJiMIrFnEECU9szsexqEth73MdV3T6W3UW0vb_U_aOAF-cqOC-6cSiU73fC0ir7MBs9ZEsLMJJcv9oa45hZVNO_4urrudzQWm-sZgLJwaaQSCbb_s1E2x97H-zo-_TBuBKj0QN2ZeeOho2-XdKPj75DRJ0Qt0t-TMqiw2Z2fjZEdO8Gy5zJQCPTKwJvzENuh0zpnQt6M4Dq74Terq-VGwQTY7KRXFuqJK

In [ ]:
temp_response.output

[ResponseReasoningItem(id='rs_01f997735c535ce3006ac796cdc79887d080e0cac18750191a', summary=[], type='reasoning', content=[], encrypted_content='gAAAAABqx5bR8vj1Sh5Oj4Xbi_TxTBa4P4oNiJrYEoLYg35EACc_g39x2HQe9J9rYz9Dq8akPu-b4d0dS2_jN3Uk9Z0JEUj-cEQb8Fj25ek7qITYpjk2AWtJGUcovQeMY9IRK9ecz5klI0sOmZcwUddLdkpPGLHvIC8iOSPg9zq144z5M9uTIfIuWyGpJuOnDYJH9zswRDAGihBS9KHdgDUkAcbhTGV6XxEChaqmRxiMs96RU8qy8uLu2Xcx9EYTR16RGCwDGjtkeNbT99zOdq4WHr7_1ZvWmYjJUa25TVZjLTzq3IU3TOl7NNNXWB-yFQd4ZhQvGIlER18ksS0TuNclg5ENTXZrgbDuFBk5NZfZFYHtk2DGoeRxnr9mQ7HvBdi4LStjDt-lK8raJe585uz-bjJiMIrFnEECU9szsexqEth73MdV3T6W3UW0vb_U_aOAF-cqOC-6cSiU73fC0ir7MBs9ZEsLMJJcv9oa45hZVNO_4urrudzQWm-sZgLJwaaQSCbb_s1E2x97H-zo-_TBuBKj0QN2ZeeOho2-XdKPj75DRJ0Qt0t-TMqiw2Z2fjZEdO8Gy5zJQCPTKwJvzENuh0zpnQt6M4Dq74Terq-VGwQTY7KRXFuqJKE6xJEmp0279lBlprHXQ5s5vTnx__EKUd7S3Ytu8iR4CnlpBGUdsbr1gtMSkIyiISu3pis-LHedFsBKS7M1TnveLxuzfHpV1HNvUvyLKB6G5U13omDKxo4XcJaeHLawtmMFvXk7r0LuQlO-x8OFlJIghd6JCo98rqiCDEZ5SkVFttYIP-JcD5EGRu-Pj9fYA-LALwRpL4eSYerFvQe2tdLVXIDwBTBG

In [ ]:
tool_calls = [x for x in temp_response.output if x.type == 'function_call']  # вземи всички items, които има за type 'function_call' => за повторно извикване на агента